In [1]:
import pandas as pd
import sqlite3

In [2]:
df = pd.read_csv('C:/Users/PROGRESSIVE/Desktop/AnalystLab Internship/Reviews.csv')
conn = sqlite3.connect('C:/Users/PROGRESSIVE/Desktop/AnalystLab Internship/airbnb.db')
df.to_sql('C:/Users/PROGRESSIVE/Desktop/AnalystLab Internship/Reviews', conn)

5373143

In [3]:
df=pd.read_sql("SELECT COUNT(*) FROM reviews",conn)
df

,COUNT(*)
0,5373143


In [4]:
# List the tables in the database
pd.read_sql("SELECT name FROM sqlite_master WHERE type = 'table'", conn)

,name
0,reviews
1,listings
2,C:/Users/PROGRESSIVE/Desktop/AnalystLab Intern...


In [5]:
# Drop the path-named copy
conn.execute('DROP TABLE "C:/Users/PROGRESSIVE/Desktop/AnalystLab Internship/Reviews"')
conn.commit()

In [6]:
# Check what's left, then count the reviews table
pd.read_sql("SELECT name FROM sqlite_master WHERE type = 'table'", conn)
pd.read_sql("SELECT COUNT(*) AS total_reviews FROM reviews", conn)

,total_reviews
0,5373143


In [7]:
pd.read_sql("SELECT COUNT(*) FROM reviews GROUP BY review_id HAVING COUNT(*) > 1", conn)

,COUNT(*)
0,2
1,2
2,2
3,2
4,2
...,...
155,2
156,2
157,2
158,2


In [8]:
pd.read_sql("""SELECT * FROM reviews WHERE review_id IN (SELECT review_id FROM reviews GROUP BY review_id HAVING COUNT(*) > 1) 
ORDER BY review_id LIMIT 20""", conn)

,listing_id,review_id,date,reviewer_id
0,2997346,200088015,2017-10-04,123346683
1,22219088,200088015,2017-10-04,123346683
2,2997346,205182329,2017-10-21,23770481
3,22219088,205182329,2017-10-21,23770481
4,2997346,223954194,2018-01-02,77129337
5,22219088,223954194,2018-01-02,77129337
6,2997346,256197483,2018-04-22,115838328
7,22219088,256197483,2018-04-22,115838328
8,466155,256940062,2018-04-24,172857558
9,22219250,256940062,2018-04-24,172857558


In [9]:
# Check if all rows vs distinct (listing_id, review_id) pairs
pd.read_sql("""SELECT (SELECT COUNT(*) FROM reviews) AS total_rows, (SELECT COUNT(*) FROM (SELECT DISTINCT listing_id, review_id FROM reviews)) AS distinct_pairs
""", conn)

,total_rows,distinct_pairs
0,5373143,5373143


In [10]:
# Check if all 160 repeated IDs share the same reviewer and date?
pd.read_sql("""
SELECT
    COUNT(*) AS repeated_ids,
    SUM(CASE WHEN n_reviewers = 1 AND n_dates = 1 THEN 1 ELSE 0 END) AS same_reviewer_and_date
FROM (
    SELECT review_id,
           COUNT(DISTINCT reviewer_id) AS n_reviewers,
           COUNT(DISTINCT date) AS n_dates
    FROM reviews
    GROUP BY review_id
    HAVING COUNT(*) > 1
)
""", conn)

,repeated_ids,same_reviewer_and_date
0,160,160


## Power BI Dashboard 1 Validation (Airbnb Market Overview)

In [11]:
# Total Listings
pd.read_sql("""SELECT DISTINCT COUNT(listing_id) FROM listings""", conn)

,COUNT(listing_id)
0,279712


In [12]:
# Number of cities represented
pd.read_sql("""SELECT COUNT(DISTINCT city) FROM Listings""", conn)

,COUNT(DISTINCT city)
0,10


In [13]:
# Listing distribution by city
pd.read_sql("""SELECT city, COUNT(*) AS total_listings FROM Listings GROUP BY city ORDER BY total_listings DESC LIMIT 10""", conn)

,city,total_listings
0,Paris,64690
1,New York,37012
2,Sydney,33630
3,Rome,27647
4,Rio de Janeiro,26615
5,Istanbul,24519
6,Mexico City,20065
7,Bangkok,19361
8,Cape Town,19086
9,Hong Kong,7087


In [14]:
# Room type distribution
pd.read_sql("""
SELECT room_type,
       COUNT(*) AS total_listings,
       ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM listings), 1) AS pct_of_listings
FROM listings
GROUP BY room_type
ORDER BY total_listings DESC
""", conn)

,room_type,total_listings,pct_of_listings
0,Entire place,182005,65.1
1,Private room,86988,31.1
2,Hotel room,5857,2.1
3,Shared room,4862,1.7


In [15]:
# Property type distribution
pd.read_sql("""SELECT property_type, COUNT(*) AS total_listings FROM Listings GROUP BY property_type ORDER BY total_listings DESC LIMIT 10""", conn)

,property_type,total_listings
0,Entire apartment,138989
1,Private room in apartment,47322
2,Private room in house,13292
3,Entire house,13273
4,Entire condominium,11250
5,Room in boutique hotel,5771
6,Entire loft,4587
7,Private room in condominium,4462
8,Private room in bed and breakfast,4238
9,Entire serviced apartment,3973


## Power BI Dashboard 2 Validation (Pricing and Property)

# Price distribution by city
pd.read_sql("""SELECT city, MIN(price) AS min_price, MAX(price) AS max_price FROM Listings WHERE price > 0 GROUP BY city""", conn)

In [17]:
# Importing the new Listings csv file to check for median, 25%, 75% using pandas
df = pd.read_csv("C:/Users/PROGRESSIVE/Desktop/AnalystLab Internship/listings_clean.csv")
df.head()

,listing_id,host_id,host_since,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_total_listings_count,host_has_profile_pic,host_identity_verified,...,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,instant_bookable,valid_price,valid_accommodates
0,281420,1466919,2011-12-03,NaN,NaN,NaN,No,1.0,Yes,No,...,100.0,10.0,10.0,10.0,10.0,10.0,10.0,No,True,True
1,3705183,10328771,2013-11-29,NaN,NaN,NaN,No,1.0,Yes,Yes,...,100.0,10.0,10.0,10.0,10.0,10.0,10.0,No,True,True
2,4082273,19252768,2014-07-31,NaN,NaN,NaN,No,1.0,Yes,No,...,100.0,10.0,10.0,10.0,10.0,10.0,10.0,No,True,True
3,4797344,10668311,2013-12-17,NaN,NaN,NaN,No,1.0,Yes,Yes,...,100.0,10.0,10.0,10.0,10.0,10.0,10.0,No,True,True
4,4823489,24837558,2014-12-14,NaN,NaN,NaN,No,1.0,Yes,No,...,100.0,10.0,10.0,10.0,10.0,10.0,10.0,No,True,True


In [18]:
df.shape

(279712, 30)

In [19]:
df["valid_price"].value_counts()

valid_price
True     279599
False       113
Name: count, dtype: int64

In [20]:
df[df["valid_price"]].groupby("city")["price"].median()

city
Bangkok           1100.0
Cape Town         1069.0
Hong Kong          386.0
Istanbul           252.0
Mexico City        664.0
New York            99.0
Paris               80.0
Rio de Janeiro     280.0
Rome                65.0
Sydney             120.0
Name: price, dtype: float64

In [21]:
df[df["valid_price"]].groupby("city")["price"].quantile(0.25)

city
Bangkok           700.0
Cape Town         650.0
Hong Kong         248.0
Istanbul          150.0
Mexico City       392.0
New York           60.0
Paris              59.0
Rio de Janeiro    157.0
Rome               45.0
Sydney             74.0
Name: price, dtype: float64

In [22]:
df[df["valid_price"]].groupby("city")["price"].quantile(0.75)

city
Bangkok           1900.0
Cape Town         2200.0
Hong Kong          700.0
Istanbul           449.0
Mexico City       1124.0
New York           151.0
Paris              120.0
Rio de Janeiro     550.0
Rome                99.0
Sydney             219.0
Name: price, dtype: float64

In [23]:
# Price variation by room type (Paris)
df[(df["valid_price"]) & (df["city"] == "Paris")].groupby("room_type")["price"].median()

room_type
Entire place     82.0
Hotel room      162.5
Private room     53.0
Shared room      34.0
Name: price, dtype: float64

In [24]:
# Price per capacity (Paris)
df[(df["valid_price"]) & (df["valid_accommodates"]) & (df["city"] == "Paris")].groupby("accommodates")["price"].median()

accommodates
1       45.0
2       69.0
3       80.0
4       99.0
5      131.0
6      168.0
7      200.0
8      250.0
9      337.5
10     312.0
11     500.0
12     480.0
13     642.0
14     566.5
15    1085.5
16     665.0
Name: price, dtype: float64

## Power BI Dashboard 3 Validation (Host Performance)

In [28]:
# Total hosts
pd.read_sql("""SELECT COUNT(DISTINCT host_id) FROM Listings""", conn)

,COUNT(DISTINCT host_id)
0,182024


In [25]:
# Listings per host
pd.read_sql("""SELECT COUNT(*) * 1.0 / COUNT(DISTINCT host_id) FROM listings""", conn)

,COUNT(*) * 1.0 / COUNT(DISTINCT host_id)
0,1.536676


In [26]:
# Host response and acceptance rate by city
pd.read_sql("""SELECT city, ROUND(AVG(host_response_rate) * 100, 0) AS avg_response_pct, ROUND(AVG(host_acceptance_rate) * 100, 0) AS avg_acceptance_pct FROM Listings GROUP BY city""", conn)

,city,avg_response_pct,avg_acceptance_pct
0,Bangkok,85.0,91.0
1,Cape Town,91.0,80.0
2,Hong Kong,83.0,80.0
3,Istanbul,83.0,85.0
4,Mexico City,89.0,89.0
5,New York,89.0,80.0
6,Paris,86.0,81.0
7,Rio de Janeiro,83.0,76.0
8,Rome,87.0,90.0
9,Sydney,88.0,79.0


In [29]:
pd.read_sql("""SELECT COUNT(*) FROM reviews WHERE date LIKE '2018-09%'""", conn)

,COUNT(*)
0,109883


In [30]:
pd.read_sql("""SELECT
    listing_id,
    COUNT(*) AS total_reviews
FROM reviews
WHERE listing_id = 2595
GROUP BY listing_id""", conn)

,listing_id,total_reviews
0,2595,48


In [33]:
pd.read_sql(""SELECT COUNT(*) AS total_reviews
FROM listings AS l
INNER JOIN reviews AS r
    ON l.listing_id = r.listing_id
WHERE l.city = 'Paris'""", conn)

,total_reviews
0,1213727


In [35]:
pd.read_sql("""
SELECT
    l.listing_id,
    l.name,
    l.city,
    r.*
FROM listings AS l
INNER JOIN reviews AS r
    ON l.listing_id = r.listing_id
WHERE l.city = 'Paris'
""", conn)

,listing_id,name,city,listing_id,review_id,date,reviewer_id
0,281420,"Beautiful Flat in le Village Montmartre, Paris",Paris,281420,172657766,2017-07-21,74346703
1,281420,"Beautiful Flat in le Village Montmartre, Paris",Paris,281420,173279221,2017-07-23,3501898
2,3705183,39 mÃÂ² Paris (Sacre CÃ âur),Paris,3705183,20792760,2014-10-05,21583508
3,3705183,39 mÃÂ² Paris (Sacre CÃ âur),Paris,3705183,32282670,2015-05-16,32285553
4,3705183,39 mÃÂ² Paris (Sacre CÃ âur),Paris,3705183,45645391,2015-09-04,34310199
...,...,...,...,...,...,...,...
1213722,38683356,Nice and cosy mini-appartement in Paris,Paris,38683356,546207032,2019-10-13,155005299
1213723,38683356,Nice and cosy mini-appartement in Paris,Paris,38683356,553159713,2019-10-23,197011717
1213724,39659000,Charming apartment near Rue Saint Maur / Oberk...,Paris,39659000,556350613,2019-10-29,102836346
1213725,40219504,Cosy apartment with view on Canal St Martin,Paris,40219504,597833586,2020-01-26,21707358
